# STUDENT INFORMATION

**Full Name:** Tăng Nhật Minh  
**Student ID:** 23127425  
**Class:** 23CNTThuc2 

# LIBRARY & CONFIGURATION

In [1]:
import json
import pandas as pd
import os
import re
import pdfplumber
import pypdf
import warnings
from openpyxl.styles import Alignment
from itertools import zip_longest

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

def get_project_paths():
    """
    Determines the input/output directories based on the current working directory.
    """
    base_dir = os.getcwd()
    raw_data_dir = os.path.join(base_dir, "Raw data")
    clean_data_dir = os.path.join(base_dir, "Clean data")
    
    # Fallback to flat structure if nested structure doesn't exist
    if not os.path.exists(raw_data_dir):
         raw_data_dir = os.path.join(base_dir, "Raw data")
         clean_data_dir = os.path.join(base_dir, "Clean data")
         
    return raw_data_dir, clean_data_dir

print("Environment setup complete. Libraries loaded.")

Environment setup complete. Libraries loaded.


# DATA CLEANING & SENTENCES SEGMENTATION

#### wikihow_vi_zh (JSON1)

In [2]:
def process_wikihow_dataset():
    """
    Task 1: Processes Wikihow JSON (ID 3184 - 3396).
    Fixes specific citation errors (e.g., [number]) and aligns sentences.
    """
    # Configuration
    INPUT_FILENAME = "wikihow_vi_zh.json"
    START_ID = 3184
    END_ID = 3396

    REGEX_GARBAGE_JSON = r'\{"smallUrl":".*?"\}'
    REGEX_ZH = r'([。！？])' 
    REGEX_VI = r'(?<=[.!?;])\s+(?=[A-ZÁÀẢÃẠĂẮẰẲẴẶÂẤẦẨẪẬĐÉÈẺẼẸÊẾỀỂỄỆÍÌỈĨỊÓÒỎÕỌÔỐỒỔỖỘƠỚỜỞỠỢÚÙỦŨỤƯỨỪỬỮỰÝỲỶỸỴ0-9])'
    REGEX_ERROR_BRACKET = r'(?<=[.!?;])\[\d+\]'

    def clean_text_wikihow(text):
        if not text: return ""
        text = re.sub(REGEX_GARBAGE_JSON, '', text)
        lines = text.split('\n')
        cleaned_lines = []
        for line in lines:
            line = re.sub(r'^[\s\-\*•]+', '', line).strip()
            line = re.sub(r'^\d+\.(\s+|$)', '', line).strip()
            if line:
                cleaned_lines.append(line)
        text = ' '.join(cleaned_lines)
        text = text.replace("'", '"') 
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def split_sentences_zh(text):
        if not text: return []
        chunks = re.split(REGEX_ZH, text)
        sentences = []
        for i in range(0, len(chunks) - 1, 2):
            sentences.append(chunks[i] + chunks[i+1])
        if len(chunks) % 2 != 0 and chunks[-1]:
            sentences.append(chunks[-1])
        return [s.strip() for s in sentences if s.strip()]

    def split_sentences_vi(text):
        if not text: return []
        sentences = re.split(REGEX_VI, text)
        return [s.strip() for s in sentences if s.strip()]

    print(f"--- STARTING TASK 1: WIKIHOW JSON (ID: {START_ID} -> {END_ID}) ---")
    
    raw_dir, clean_dir = get_project_paths()
    input_path = os.path.join(raw_dir, INPUT_FILENAME)
    
    if not os.path.exists(clean_dir): os.makedirs(clean_dir)
    if not os.path.exists(input_path):
        print(f"ERROR: File not found {input_path}"); return

    try:
        with open(input_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception as e:
        print(f"JSON Read Error: {e}"); return

    # --- STEP 1: Initial Processing ---
    temp_rows = [] 
    target_data = [item for item in data]
    
    for item in target_data:
        try:
            current_id = int(item.get('id', -1))
        except: continue
        
        if not (START_ID <= current_id <= END_ID): continue
        
        zh_raw = item.get('cn', item.get('zh', '')) 
        vi_raw = item.get('vi', '')
        
        zh_clean = clean_text_wikihow(zh_raw)
        vi_clean = clean_text_wikihow(vi_raw)
        
        vi_sents = split_sentences_vi(vi_clean)
        zh_sents = split_sentences_zh(zh_clean)
        
        pairs = list(zip_longest(zh_sents, vi_sents, fillvalue=''))
        
        for z, v in pairs:
            temp_rows.append({
                'src_id': current_id, 
                'src_lang': z, 
                'tgt_lang': v
            })

    if not temp_rows:
        print("No data in ID range!"); return

    # --- STEP 2: Post-Processing (Fixing Bracket Errors) ---
    final_rows = []

    for row in temp_rows:
        vi_text = row['tgt_lang']
        current_id = row['src_id']
        current_zh = row['src_lang']

        # Check for citation errors
        if vi_text and re.search(REGEX_ERROR_BRACKET, vi_text):
            
            # Remove [number] and 'Nguồn nghiên cứu'
            clean_vi_again = re.sub(r'\[\d+\](?:\s*Nguồn nghiên cứu)?', '', vi_text, flags=re.IGNORECASE)
            
            # Resplit sentences
            sub_parts = split_sentences_vi(clean_vi_again)
            
            if len(sub_parts) > 1:
                # First line: Keep ID + Chinese
                final_rows.append({
                    'src_id': current_id,
                    'src_lang': current_zh,
                    'tgt_lang': sub_parts[0]
                })
                # Subsequent lines: Keep ID + Empty Chinese
                for part in sub_parts[1:]:
                    final_rows.append({
                        'src_id': current_id,
                        'src_lang': '', 
                        'tgt_lang': part
                    })
            else:
                row['tgt_lang'] = clean_vi_again
                final_rows.append(row)
        else:
            final_rows.append(row)

    # --- STEP 3: Export CSV ---
    df = pd.DataFrame(final_rows)
    
    src_name = os.path.splitext(INPUT_FILENAME)[0]
    output_filename = f"{src_name}_{START_ID}_{END_ID}.csv" 
    output_path = os.path.join(clean_dir, output_filename)
    
    try:
        df.to_csv(output_path, index=False, encoding='utf-8-sig')
        
        print(f"Successfully cleaned to {output_filename}") 
    except PermissionError:
        print("ERROR: Please close the file before running!")

#### JSON2_multimed_18504 (JSON 2)

In [3]:
def process_multimedia_dataset():
    """
    Task 2: Processes multimedia JSON data (ID 17369 - 18504).
    Aligns Vietnamese and Chinese sentences using regex splitting.
    """
    # Configuration
    INPUT_FILENAME = "JSON2_multimed_18504.json"
    START_ID = 17369
    END_ID = 18504
    
    # Regex for sentence splitting
    REGEX_ZH = r'([。！？])' 
    REGEX_VI = r'(?<=[.!?;])\s+(?=[A-ZÁÀẢÃẠĂẮẰẲẴẶÂẤẦẨẪẬĐÉÈẺẼẸÊẾỀỂỄỆÍÌỈĨỊÓÒỎÕỌÔỐỒỔỖỘƠỚỜỞỠỢÚÙỦŨỤƯỨỪỬỮỰÝỲỶỸỴ])'

    def split_sentences_zh(text):
        if not text: return []
        chunks = re.split(REGEX_ZH, text)
        sentences = []
        for i in range(0, len(chunks) - 1, 2):
            sentences.append(chunks[i] + chunks[i+1])
        if len(chunks) % 2 != 0 and chunks[-1]:
            sentences.append(chunks[-1])
        return [s.strip() for s in sentences if s.strip()]

    def split_sentences_vi(text):
        if not text: return []
        sentences = re.split(REGEX_VI, text)
        return [s.strip() for s in sentences if s.strip()]

    print(f"--- STARTING TASK 2: MULTIMEDIA JSON (ID: {START_ID} -> {END_ID}) ---")
    
    raw_dir, clean_dir = get_project_paths()
    input_path = os.path.join(raw_dir, INPUT_FILENAME)
    
    if not os.path.exists(clean_dir):
        os.makedirs(clean_dir)

    if not os.path.exists(input_path):
        print(f"ERROR: File not found at {input_path}")
        return

    try:
        with open(input_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception as e:
        print(f"JSON Read Error: {e}")
        return

    processed_rows = []

    for item in data:
        # Filter by ID
        try:
            current_id = int(item.get('id', -1))
        except:
            continue 
            
        if not (START_ID <= current_id <= END_ID):
            continue 

        vi_text = item.get('vi', '')
        zh_text = item.get('zh', '')
        
        # Sentence Segmentation
        vi_sents = split_sentences_vi(vi_text)
        zh_sents = split_sentences_zh(zh_text)
        
        # Alignment Logic
        if len(vi_sents) == len(zh_sents) and len(vi_sents) > 0:
            for v, z in zip(vi_sents, zh_sents):
                processed_rows.append({
                    'src_id': current_id,
                    'src_lang': z,  # Chinese
                    'tgt_lang': v   # Vietnamese
                })
        else:
            # Fallback: Keep original paragraph if counts mismatch
            processed_rows.append({
                'src_id': current_id,
                'src_lang': zh_text,
                'tgt_lang': vi_text
            })

    # Export to Excel
    if not processed_rows:
        print("No data found in this ID range.")
        return

    df = pd.DataFrame(processed_rows)
    src_name = os.path.splitext(INPUT_FILENAME)[0]
    output_filename = f"{src_name}_{START_ID}_{END_ID}.csv" 
    output_path = os.path.join(clean_dir, output_filename)
    
    df.to_csv(output_path, index=False, encoding='utf-8-sig')

    print(f"Successfully cleaned to {output_filename}")

#### PDF2_1001 Buc Thu Viet Cho Tuong Lai - Nhieu Tac Gia (PDF2)

In [4]:
def process_future_letters_dataset():
    """
    Task 3: Processes '1001 Letters to the Future' PDF (ID 876 - 1001).
    Handles TOC removal and garbage cleaning using pdfplumber.
    """
    # Configuration
    INPUT_FILENAME = "PDF2_1001 Buc Thu Viet Cho Tuong Lai - Nhieu Tac Gia.pdf"
    START_ID = 876
    END_ID = 1001
    START_PAGE_INDEX = 350 
    END_PAGE_INDEX = 430

    REGEX_ID_SEARCH = r"\b(87[6-9]|8[8-9]\d|9\d{2}|1000|1001)\b"
    REGEX_VN_START = r"(?m)^\s*[a-zA-ZÀ-ỹ\"“].*?$"
    REGEX_ZH_SENT = r'([。！？\?])' 
    REGEX_VI_SENT = r'(?<=[.!?;])\s+(?=[A-ZÁÀẢÃẠĂẮẰẲẴẶÂẤẦẨẪẬĐÉÈẺẼẸÊẾỀỂỄỆÍÌỈĨỊÓÒỎÕỌÔỐỒỔỖỘƠỚỜỞỠỢÚÙỦŨỤƯỨỪỬỮỰÝỲỶỸỴ0-9\"“])'
    REGEX_TOC_CUT = r"(1001\s+BỨC\s+THƯ|MMụụcc\s+llụụcc|Mục\s+lục|dtv-ebook)"

    DELETE_PATTERNS = [
        r"11000011",
        r"BBỨỨCC\s+TTHHƯƯ\s+VVIIẾẾTT\s+CCHHOO",
        r"TTƯƯƠƠNNGG\s+LLAAII",
        r"NNhhiiềềuu\s+TTáácc\s+GGiiảả",
        r"ddttvv--eebbooookk\.\.ccoomm",
        r"\d{6}--\d{6}",
        r"--- PAGE \d+ ---"
    ]

    def split_sentences_zh(text):
        if not text: return []
        chunks = re.split(REGEX_ZH_SENT, text)
        sentences = []
        for i in range(0, len(chunks) - 1, 2):
            sentences.append(chunks[i] + chunks[i+1])
        if len(chunks) % 2 != 0 and chunks[-1]:
            sentences.append(chunks[-1])
        return [s.strip() for s in sentences if s.strip()]

    def split_sentences_vi(text):
        if not text: return []
        sentences = re.split(REGEX_VI_SENT, text)
        return [s.strip() for s in sentences if s.strip()]

    def clean_garbage(text):
        for pat in DELETE_PATTERNS:
            text = re.sub(pat, "", text)
        return text.strip()

    print(f"--- STARTING TASK 3: PDF LETTERS (ID: {START_ID} -> {END_ID}) ---")
    
    raw_dir, clean_dir = get_project_paths()
    input_path = os.path.join(raw_dir, INPUT_FILENAME)
    
    if not os.path.exists(clean_dir): os.makedirs(clean_dir)
    if not os.path.exists(input_path):
        print(f"ERROR: File not found {input_path}"); return

    # 1. Read PDF Content
    full_text = ""
    try:
        with pdfplumber.open(input_path) as pdf:
            max_pages = len(pdf.pages)
            start = max(0, START_PAGE_INDEX)
            end = min(max_pages, END_PAGE_INDEX)
            for i in range(start, end):
                t = pdf.pages[i].extract_text()
                if t: full_text += "\n" + t
    except Exception as e:
        print(f"Error: {e}"); return

    # 2. Process Content
    full_text = clean_garbage(full_text)
    matches = list(re.finditer(REGEX_ID_SEARCH, full_text))
    processed_rows = []

    for i in range(len(matches)):
        current_id = int(matches[i].group())
        start_pos = matches[i].end()
        end_pos = matches[i+1].start() if i < len(matches) - 1 else len(full_text)
            
        content_block = full_text[start_pos:end_pos].strip()
        
        # Cut off Table of Contents / Garbage
        toc_match = re.search(REGEX_TOC_CUT, content_block, flags=re.IGNORECASE)
        if toc_match:
            content_block = content_block[:toc_match.start()].strip()
        
        # Split Chinese / Vietnamese
        split_match = re.search(REGEX_VN_START, content_block)
        if split_match:
            split_idx = split_match.start()
            zh_text = content_block[:split_idx].strip()
            vi_text = content_block[split_idx:].strip()
        else:
            zh_text = content_block
            vi_text = ""

        # Segmentation
        zh_sents = split_sentences_zh(zh_text)
        vi_sents = split_sentences_vi(vi_text)
        
        # Alignment (zip_longest)
        pairs = list(zip_longest(zh_sents, vi_sents, fillvalue=''))
        
        for z, v in pairs:
            processed_rows.append({
                'src_id': current_id, 
                'src_lang': z, 
                'tgt_lang': v
            })

    # 3. Export CSV
    if not processed_rows: print("No data processed."); return

    df = pd.DataFrame(processed_rows)
    df = df[['src_id', 'src_lang', 'tgt_lang']]
    
    src_name = os.path.splitext(INPUT_FILENAME)[0]
    output_filename = f"{src_name}_{START_ID}_{END_ID}.csv" 
    output_path = os.path.join(clean_dir, output_filename)
    
    df.to_csv(output_path, index=False, encoding='utf-8-sig')

    print(f"Successfully cleaned to {output_filename}")

#### PDF3_vui học tiếng trung qua truyện cười (PDF3)

In [5]:
def process_funny_stories_dataset():
    """
    Task 4: Processes 'Funny Stories' PDF (ID 76 - 100).
    Uses dual-regex to catch missing IDs and performs surgical text cleaning.
    """
    # Configuration
    INPUT_FILENAME = "PDF3_vui học tiếng trung qua truyện cười.pdf"
    START_ID = 76
    END_ID = 100
    START_PAGE_INDEX = 160 
    END_PAGE_INDEX = 250

    # Advanced Regex for Dual ID detection
    REGEX_DUAL_ID = r"(?<!\d)(7[6-9]|[89]\d|100)(?=\s*[\u4e00-\u9fff])|(?<=[\u4e00-\u9fff])\s*(7[6-9]|[89]\d|100)(?!\d)"
    VN_PATTERN_FRIEND = r"[a-zA-ZÀ-ỹ]+(?=\s+[a-zA-ZÀ-ỹ])"
    REGEX_ZH_SENT = r'([。！？\?])' 
    REGEX_VI_SENT = r'(?<=[.!?;])\s+'

    def has_chinese_char(text):
        return re.search(r"[\u4e00-\u9fff]", text) is not None

    def clean_vi_text_surgical(text):
        """
        Surgical text cleaning logic (Version 12)
        """
        lines = text.split('\n')
        clean_lines = []
        
        STOP_KEYWORDS = [
            "词语表", "TỪ MỚI", "TỪ NGỮ", "GIẢI THÍCH", "Phiên âm", 
            "TRỌNG ĐIỂM", "Mẫu câu", "LỜI BÌNH", "MCBooks", "Chuyên sách"
        ]
        
        for line in lines:
            s = line.strip()
            if not s: continue
            
            # 1. Stop Check (Cut tail)
            is_stop = False
            for kw in STOP_KEYWORDS:
                if kw.lower() in s.lower():
                    is_stop = True
                    break
            if is_stop: break 
                
            if has_chinese_char(s): break 
                
            # 2. Skip Check
            if re.match(r"^\d+$", s): continue # Page number
            if s.isupper() and len(s) > 3: continue # Uppercase Headers
                
            clean_lines.append(s)
            
        return "\n".join(clean_lines).strip()

    def split_sentences_zh(text):
        if not text: return []
        chunks = re.split(REGEX_ZH_SENT, text)
        sentences = [chunks[i] + chunks[i+1] for i in range(0, len(chunks) - 1, 2)]
        if len(chunks) % 2 != 0 and chunks[-1]: sentences.append(chunks[-1])
        return [s.strip() for s in sentences if s.strip()]

    def split_sentences_vi(text):
        if not text: return []
        return [s.strip() for s in re.split(REGEX_VI_SENT, text) if s.strip()]

    print(f"--- STARTING TASK 4: FUNNY STORIES PDF (ID: {START_ID} -> {END_ID}) ---")
    
    raw_dir, clean_dir = get_project_paths()
    input_path = os.path.join(raw_dir, INPUT_FILENAME)
    
    if not os.path.exists(clean_dir): os.makedirs(clean_dir)
    
    full_text = ""
    try:
        with open(input_path, 'rb') as f:
            reader = pypdf.PdfReader(f, strict=False)
            max_pages = len(reader.pages)
            for i in range(START_PAGE_INDEX, min(max_pages, END_PAGE_INDEX)):
                t = reader.pages[i].extract_text()
                if t: full_text += "\n" + t
    except Exception as e:
        print(f"File Read Error: {e}"); return

    # Find IDs using Dual Regex
    matches = list(re.finditer(REGEX_DUAL_ID, full_text))
    
    processed_rows = []
    processed_ids = set()

    for i in range(len(matches)):
        m = matches[i]
        current_id = int(m.group(1) if m.group(1) else m.group(2))
        
        # Avoid processing same ID twice (unless ID is 100 which might require special handling, logic kept as original)
        if current_id in processed_ids and current_id != 100: 
            continue
        processed_ids.add(current_id)
        
        start_pos = m.end()
        end_pos = matches[i+1].start() if i < len(matches) - 1 else len(full_text)
            
        raw_block = full_text[start_pos:end_pos]
        raw_block = raw_block.replace('"', '').replace('“', '').replace('”', '')
        
        vn_match = re.search(VN_PATTERN_FRIEND, raw_block)
        
        if vn_match:
            match_pos = vn_match.start()
            line_start = raw_block.rfind("\n", 0, match_pos)
            if line_start == -1: line_start = 0
            else: line_start += 1 
                
            zh_text = raw_block[:line_start].strip()
            vi_text_raw = raw_block[line_start:].strip()
            
            # Surgical Cleaning
            vi_text = clean_vi_text_surgical(vi_text_raw)
            
        else:
            zh_text = raw_block.strip()
            vi_text = ""

        # Alignment
        zh_sents = split_sentences_zh(zh_text)
        vi_sents = split_sentences_vi(vi_text)
        
        pairs = list(zip_longest(zh_sents, vi_sents, fillvalue=''))
        for z, v in pairs:
            processed_rows.append({'src_id': current_id, 'src_lang': z, 'tgt_lang': v})

    # Export File
    if not processed_rows: print("No data to export."); return

    df = pd.DataFrame(processed_rows)
    df = df[['src_id', 'src_lang', 'tgt_lang']]
    df = df.sort_values(by='src_id')
    
    src_name = os.path.splitext(INPUT_FILENAME)[0]
    output_filename = f"{src_name}_{START_ID}_{END_ID}.csv" 
    output_path = os.path.join(clean_dir, output_filename)
    
    try:
        df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"Successfully cleaned to {output_filename}")
    except PermissionError:
        print("ERROR: Please close the file before running!")

# MAIN

In [6]:
if __name__ == "__main__":
    # 1. Run Wikihow Task
    process_wikihow_dataset()
    print("-" * 60 + "\n")

    # 2. Run Multimedia Task
    process_multimedia_dataset()
    print("-" * 60 + "\n")

    # 3. Run Future Letters Task
    process_future_letters_dataset()
    print("-" * 60 + "\n")

    # 4. Run Funny Stories Task
    process_funny_stories_dataset()

--- STARTING TASK 1: WIKIHOW JSON (ID: 3184 -> 3396) ---
Successfully cleaned to wikihow_vi_zh_3184_3396.csv
------------------------------------------------------------

--- STARTING TASK 2: MULTIMEDIA JSON (ID: 17369 -> 18504) ---
Successfully cleaned to JSON2_multimed_18504_17369_18504.csv
------------------------------------------------------------

--- STARTING TASK 3: PDF LETTERS (ID: 876 -> 1001) ---
Successfully cleaned to PDF2_1001 Buc Thu Viet Cho Tuong Lai - Nhieu Tac Gia_876_1001.csv
------------------------------------------------------------

--- STARTING TASK 4: FUNNY STORIES PDF (ID: 76 -> 100) ---
Successfully cleaned to PDF3_vui học tiếng trung qua truyện cười_76_100.csv
